In [ ]:
import duckdb
import os
from dotenv import load_dotenv

load_dotenv("../.env")

conn = duckdb.connect()
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

s3_endpoint = "localhost:9000"
access_key = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")

conn.execute(f"""
CREATE SECRET (
    TYPE S3,
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    ENDPOINT '{s3_endpoint}',
    URL_STYLE 'path',
    USE_SSL 'false'
);
""")

print("Connected to MinIO")

print("\nYahoo raw data (AAPL 1h):")
df_yahoo = conn.execute(
    "SELECT * FROM read_parquet('s3://raw-data/AAPL_1h.parquet') LIMIT 5"
).df()
display(df_yahoo)

print("\nBybit raw data (BTCUSDT 1h):")
df_bybit = conn.execute(
    "SELECT * FROM read_parquet('s3://raw-data/BTCUSDT_1h.parquet') LIMIT 5"
).df()
display(df_bybit)

conn_db = duckdb.connect("../data/analytics/financial_data.duckdb", read_only=True)

print("\nYahoo clean data by interval:")
yf_intervals = conn_db.execute(
    "SELECT DISTINCT interval FROM clean_yahoo_stocks ORDER BY interval"
).df()["interval"].tolist()

for interval in yf_intervals:
    print(f"\nInterval: {interval}")
    query = f"SELECT * FROM clean_yahoo_stocks WHERE interval = '{interval}' ORDER BY date LIMIT 5"
    display(conn_db.execute(query).df())

print("\nBybit clean data by interval:")
bybit_intervals = conn_db.execute(
    "SELECT DISTINCT interval FROM clean_bybit_crypto ORDER BY interval"
).df()["interval"].tolist()

for interval in bybit_intervals:
    print(f"\nInterval: {interval}")
    query = f"SELECT * FROM clean_bybit_crypto WHERE interval = '{interval}' ORDER BY date LIMIT 5"
    display(conn_db.execute(query).df())

conn_db.close()

In [ ]:
import duckdb

conn_db = duckdb.connect('../data/analytics/financial_data.duckdb', read_only=True)

print("==================================================")
print("🥇 UNIFIED GOLD LAYER (Local DuckDB Table) 🥇")
print("==================================================")

try:
    query = """
        SELECT * 
        FROM gold_financial_analytics
        WHERE asset_symbol IN ('AAPL', 'BTCUSDT') AND interval = '1d'
        ORDER BY asset_class, date DESC
        LIMIT 10
    """
    df_gold = conn_db.execute(query).df()
    
    display(df_gold)
    
except Exception as e:
    print(f"Error reading Gold table: {e}")

conn_db.close()

🥇 UNIFIED GOLD LAYER (Local DuckDB Table) 🥇


,asset_symbol,asset_class,interval,date,open,high,low,close,volume,daily_volatility,sma_7,sma_30
0,BTCUSDT,Crypto,1d,2026-03-06,70847.6,71384.5,68130.1,68329.1,73670.220,3254.4,68800.428571,67910.923333
1,BTCUSDT,Crypto,1d,2026-03-05,72640.4,73537.9,70610.0,70847.6,104007.762,2927.9,68443.900000,68070.963333
2,BTCUSDT,Crypto,1d,2026-03-04,68299.5,74021.8,67348.4,72640.4,143556.262,6673.4,67958.171429,68233.483333
3,BTCUSDT,Crypto,1d,2026-03-03,68799.9,69222.0,66054.3,68299.5,112639.401,3167.7,67289.300000,68435.266667
4,BTCUSDT,Crypto,1d,2026-03-02,65746.9,70066.0,65238.5,68799.9,116572.400,4827.5,66679.728571,68723.233333
5,BTCUSDT,Crypto,1d,2026-03-01,66939.6,68183.2,65001.0,65746.9,90228.941,3182.2,66081.928571,69053.383333
6,BTCUSDT,Crypto,1d,2026-02-28,65833.4,67739.5,62972.3,66939.6,119624.753,4767.2,66348.157143,69669.236667
7,BTCUSDT,Crypto,1d,2026-02-27,67447.5,68185.6,64888.0,65833.4,82996.047,3297.6,66490.285714,70257.910000
8,BTCUSDT,Crypto,1d,2026-02-26,67958.3,68841.7,66451.1,67447.5,90292.374,2390.6,66798.257143,71038.853333
9,BTCUSDT,Crypto,1d,2026-02-25,64032.5,69989.8,63869.5,67958.3,148559.208,6120.3,66731.771429,71764.013333
